# Pretraining on the sharded Hugging Face dataset

`williamalxndr/chess-pretrain-data` holds one shard per PGN file
(`{name}_states.pt`, `{name}_masks.pt`, `{name}_targets.pt`), ~68 GB in total.
Shards are streamed: each is downloaded, trained over, then deleted before the
next one is fetched, so peak disk is a single shard.

In [ ]:
import os
import sys
import subprocess

IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ or os.path.exists("/kaggle/input")

REPO_URL = "https://github.com/williamalxndr/chess-engine.git"
REPO_BRANCH = "master"

if IS_KAGGLE:
    repo_dir = "/kaggle/working/chess-engine"
    if not os.path.exists(repo_dir):
        subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, repo_dir], check=True)
    else:
        subprocess.run(["git", "-C", repo_dir, "fetch", "origin", REPO_BRANCH], check=True)
        subprocess.run(["git", "-C", repo_dir, "reset", "--hard", f"origin/{REPO_BRANCH}"], check=True)

    os.chdir(repo_dir)
    print("code:", subprocess.run(["git", "log", "-1", "--oneline"],
                                  capture_output=True, text=True).stdout.strip())
    subprocess.run(["pip", "install", "-r", "requirements.txt"], check=True)
    sys.path.append(".")
else:
    sys.path.append("..")

import numpy as np
import torch

In [ ]:
# The pretrain dataset repo is private, so every hub call needs a token.
# huggingface_hub reads HF_TOKEN from the environment automatically.
if IS_KAGGLE:
    from kaggle_secrets import UserSecretsClient

    user_secrets = UserSecretsClient()
    os.environ["HF_TOKEN"] = user_secrets.get_secret("HF_TOKEN")

print("HF_TOKEN set:", bool(os.environ.get("HF_TOKEN")))

In [3]:
DEVICE = torch.accelerator.current_accelerator() if torch.accelerator.is_available() else "cpu"
DEVICE

device(type='mps')

In [5]:
from pretrain.shards import DEFAULT_REPO_ID, list_shards, split_shards

REPO_ID = DEFAULT_REPO_ID
SHARD_DIR = "/kaggle/working/shards" if IS_KAGGLE else "shards"

# The full dataset does not fit on a Kaggle disk; keep only the shard in use.
KEEP_FILES = False

ALL_SHARDS = list_shards(REPO_ID)
TRAIN_SHARDS, TEST_SHARDS = split_shards(ALL_SHARDS, holdout=2, seed=0)

print(f"{len(ALL_SHARDS)} shards on {REPO_ID}")
print(f"train: {len(TRAIN_SHARDS)} shards")
print(f"test : {TEST_SHARDS}")

56 shards on williamalxndr/chess-pretrain-data
train: 54 shards
test : ['Kasparov', 'PCAChamp1995']


In [ ]:
# Row counts come from the *_targets.pt files only (a few MB each), so this is
# cheap relative to the dataset. Needed only to size the LR schedule when
# training by num_epoch; skip it when training by duration_hour.
from pretrain.shards import shard_row_counts

COUNT_ROWS = False

ROW_COUNTS = shard_row_counts(ALL_SHARDS, REPO_ID, local_dir=SHARD_DIR) if COUNT_ROWS else None
TRAIN_ROWS = sum(ROW_COUNTS[name] for name in TRAIN_SHARDS) if ROW_COUNTS else None

print(f"train rows: {TRAIN_ROWS:,}" if TRAIN_ROWS else "row counts not computed")

In [ ]:
from core import factory

# DataParallel replicates from device_ids[0], so the module must already live
# on the accelerator before it is wrapped (and before the optimizer is built,
# since fused AdamW validates param devices at construction time).
network = factory.build_network("chess").to(DEVICE)

In [ ]:
from torch.optim import Adam, AdamW
from torch.nn import CrossEntropyLoss, MSELoss

optimizer = AdamW(network.parameters(), lr=3e-4, fused=True)
value_loss_fn = MSELoss()
policy_loss_fn = CrossEntropyLoss()

In [ ]:
from pathlib import Path

from huggingface_hub import hf_hub_download
from torch.nn.parallel import DataParallel

from core.network import PolicyValueNetwork


def unwrap(network):
    return network.module if isinstance(network, DataParallel) else network


def save_network(network, game: str = "chess", version: str = "v2", file_name: str = "example", parent_dir: str = "checkpoints", path: str = None, push_to_hf: bool = False, repo_id: str = None):
    unwrap(network).save(game=game, version=version, file_name=file_name, parent_dir=parent_dir, path=path, push_to_hf=push_to_hf, repo_id=repo_id)


def save_training_state(path, network, best_network, optimizer, scheduler,
                        step, epoch, consumed, total_steps,
                        push_to_hf=False, repo_id=None, path_in_repo=None):
    """Write everything a later session needs to continue this run.

    Weights alone are not enough: without the AdamW moments the optimiser has
    to re-estimate them, and without the scheduler/step the LR schedule
    restarts from warmup. `consumed` is what stops a new session from
    re-training shards this epoch has already seen.
    """
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    torch.save({
        "network":      unwrap(network).state_dict(),
        "best_network": unwrap(best_network).state_dict(),
        "optimizer":    optimizer.state_dict(),
        "scheduler":    scheduler.state_dict(),
        "step":         step,
        "epoch":        epoch,
        "consumed":     sorted(consumed),
        "total_steps":  total_steps,
    }, path)

    if push_to_hf:
        PolicyValueNetwork.push_to_hub(file_path=str(path), repo_id=repo_id, path=path_in_repo)

    return path


def load_training_state(path_in_repo, repo_id, local_path=None):
    """Return the saved state, or None when this is a fresh run.

    A local file wins over the Hub copy: on Kaggle /kaggle/working can be
    carried into the next session as notebook output, which is faster than
    re-downloading.
    """
    if local_path and Path(local_path).exists():
        print(f"resuming from local {local_path}")
        # weights_only=False: the file holds optimiser/scheduler bookkeeping,
        # not just tensors. Only load state files you produced yourself.
        return torch.load(local_path, map_location="cpu", weights_only=False)

    try:
        downloaded = hf_hub_download(repo_id=repo_id, filename=path_in_repo, repo_type="model")
    except Exception as error:
        print(f"no saved state on {repo_id} ({type(error).__name__}); starting fresh")
        return None

    print(f"resuming from {repo_id}/{path_in_repo}")
    return torch.load(downloaded, map_location="cpu", weights_only=False)

## Sanity check

Channel 20 of the v1 encoder stores the legal-move count over 218; it should
agree with the popcount of the legal mask.

In [ ]:
from pretrain.shards import load_shard, release_shard

_shard = load_shard(TEST_SHARDS[0], REPO_ID, local_dir=SHARD_DIR)
print(f"{_shard.name}: states={tuple(_shard.states.shape)} masks={tuple(_shard.masks.shape)}")

mask_count    = _shard.masks[:4096].sum(dim=1).float()
channel_count = _shard.states[:4096, 20, 0, 0] * 218

avg_legal_mask    = mask_count.mean().item()
avg_legal_channel = channel_count.mean().item()

print("avg legal moves (from mask)   :", avg_legal_mask)
print("avg legal moves (from ch. 20) :", avg_legal_channel)
print("max |diff| across samples     :", (mask_count - channel_count).abs().max().item())

print("uniform-over-legal baseline (mask)   :", np.log(avg_legal_mask))
print("uniform-over-legal baseline (ch. 20) :", np.log(avg_legal_channel))

release_shard(_shard, delete_files=not KEEP_FILES)

In [ ]:
import itertools
import random
import time

import torch
from torch import optim
from torch.optim import lr_scheduler
from torch.utils.data import TensorDataset, DataLoader
from torch.nn import DataParallel

from core.network import PolicyValueNetwork
from pretrain.shards import iter_shards
import copy
from training.evaluation import evaluate as match_evaluate


def build_eval_tensors(shard_names, eval_samples=4096, repo_id=None, shard_dir=None,
                       keep_files=False, device="cpu"):
    """Materialise a fixed evaluation set from the held-out shards.

    Rows are copied off the mmap because each shard's file is deleted as soon
    as the next one is fetched.
    """
    repo_id = repo_id or REPO_ID
    shard_dir = shard_dir or SHARD_DIR

    per_shard = max(1, eval_samples // len(shard_names))
    states, masks, policy, value = [], [], [], []

    for shard in iter_shards(shard_names, repo_id=repo_id, local_dir=shard_dir,
                             keep_files=keep_files):
        n = min(len(shard), per_shard)
        states.append(shard.states[:n].clone())
        masks.append(shard.masks[:n].clone())
        policy.append(shard.policy[:n].clone())
        value.append(shard.value[:n].clone())

    return (
        torch.cat(states).to(device),
        torch.cat(policy).to(device),
        torch.cat(value).unsqueeze(-1).to(device),
        torch.cat(masks).to(device),
    )


def evaluate(network, test_states, test_policy, test_value, test_masks,
             policy_loss_fn, value_loss_fn, batch_size=256):
    network.eval()
    total_policy_loss, total_value_loss, n_batches = 0.0, 0.0, 0

    with torch.no_grad():
        for i in range(0, len(test_states), batch_size):
            batch_states = test_states[i:i+batch_size]
            batch_policy = test_policy[i:i+batch_size]
            batch_value  = test_value[i:i+batch_size]
            batch_mask   = test_masks[i:i+batch_size]

            policy_head, value_head = network(batch_states)
            policy_head = policy_head.masked_fill(~batch_mask.to(policy_head.device), float("-inf"))

            total_policy_loss += policy_loss_fn(policy_head, batch_policy).item()
            total_value_loss  += value_loss_fn(value_head, batch_value).item()
            n_batches += 1

    network.train()

    return total_policy_loss / n_batches, total_value_loss / n_batches


def train(network: PolicyValueNetwork,
          optimizer: optim.Optimizer,
          train_shards: list[str],
          policy_loss_fn,
          value_loss_fn,
          test_shards: list[str] | None = None,
          batch_size: int = 256,
          num_epoch: int | None = None,
          duration_hour: float | None = None,
          total_hour: float | None = None,
          train_rows: int | None = None,
          save_path: str | None = "default.pt",
          repo_id: str | None = None,
          shard_dir: str | None = None,
          keep_files: bool = False,
          num_workers: int = 2,
          eval_samples: int = 4096,
          seed: int = 0,
          log_every: int = 50,
          game: str = "chess",
          version: str = "v2",
          file_name: str = "pretrained",
          parent_dir: str = "checkpoints",
          push_to_hf: bool = True,
          push_repo_id: str = "williamalxndr/chess-trained-network",
          eval_games: int = 4,
          eval_rollout: int = 100,
          win_rate_threshold: float = 0.55,
          resume: bool = True,
          state_path: str = "pretrain_state.pt",
          checkpoint_every_min: float = 20):
    """Train over shards streamed from the Hub, one shard resident at a time.

    The run survives a Kaggle session ending: `duration_hour` bounds *this*
    session while `total_hour` describes the whole multi-session run, so the
    LR schedule spans all of it instead of restarting every session. Shards
    finished in the current epoch are recorded and skipped on resume.

    Nothing reaches the Hub on training progress alone: both the weights and
    the resume state are uploaded only after the current net beats the running
    best in a head-to-head match. Every checkpoint is still written locally.
    """

    if num_epoch is None and duration_hour is None:
        raise ValueError("Must specify at least one of num_epoch or duration_hour")

    repo_id = repo_id or REPO_ID
    shard_dir = shard_dir or SHARD_DIR
    state_in_repo = f"{game}/{version}/{file_name}_state.pt"

    start = time.time()
    step = 0
    start_epoch = 0
    consumed: set[str] = set()
    saved_total_steps = None

    # Continue the previous session if a state file exists.
    state = load_training_state(state_in_repo, push_repo_id, local_path=state_path) if resume else None
    if state:
        network.load_state_dict(state["network"])
        optimizer.load_state_dict(state["optimizer"])
        step = state["step"]
        start_epoch = state["epoch"]
        consumed = set(state["consumed"])
        saved_total_steps = state.get("total_steps")
        print(f"resumed: step {step}, epoch {start_epoch}, "
              f"{len(consumed)}/{len(train_shards)} shards already done this epoch")

    # Distributed training
    if torch.cuda.is_available():
        network = DataParallel(network)

    # Running best for head-to-head promotion; deepcopy the unwrapped module.
    best_network = copy.deepcopy(unwrap(network))
    if state:
        best_network.load_state_dict(state["best_network"])

    # Held-out evaluation set, built once and kept resident.
    has_test = bool(test_shards)
    if has_test:
        test_states, test_policy, test_value, test_masks = build_eval_tensors(
            test_shards, eval_samples=eval_samples, repo_id=repo_id,
            shard_dir=shard_dir, keep_files=keep_files, device=DEVICE,
        )
        print(f"eval set: {len(test_states)} positions from {test_shards}")

    # The step count cannot be derived from a dataloader here, since no shard is
    # loaded yet. Use the exact row count when training by epochs, and the
    # measured ~11.2k steps/hour otherwise. A resumed run keeps the horizon it
    # was planned with, so the cosine curve stays continuous across sessions.
    if saved_total_steps:
        total_steps = saved_total_steps
    elif num_epoch is not None:
        if train_rows is None:
            raise ValueError(
                "num_epoch needs train_rows; run the row-count cell with COUNT_ROWS = True "
                "or train by duration_hour instead"
            )
        total_steps = max(1, (train_rows // batch_size) * num_epoch)
    else:
        total_steps = max(2, int(11_200 * (total_hour or duration_hour)))

    warmup_steps = max(1, min(1000, total_steps // 20))
    if total_steps <= warmup_steps:
        warmup_steps = max(1, total_steps - 1)

    warmup = lr_scheduler.LinearLR(optimizer, start_factor=0.01, total_iters=warmup_steps)
    cosine = lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps - warmup_steps, eta_min=3e-5)
    scheduler = lr_scheduler.SequentialLR(optimizer, schedulers=[warmup, cosine], milestones=[warmup_steps])

    if state:
        scheduler.load_state_dict(state["scheduler"])

    print(f"planned steps: {total_steps} (warmup {warmup_steps}) | {total_steps - step} left")

    epoch_iter = range(start_epoch, num_epoch) if num_epoch is not None else itertools.count(start_epoch)
    epoch = start_epoch   # the loop below may not run at all on an already-finished run
    out_of_time = False
    promoted = False      # gates every push to the Hub; no match has been played yet
    last_checkpoint = time.time()

    save_network(network, path=save_path)

    for epoch in epoch_iter:
        if out_of_time:
            break

        # Shards already trained in this epoch are dropped, so a resumed
        # session picks up the remainder instead of starting the epoch over.
        order = [name for name in train_shards if name not in consumed]
        random.Random(seed + epoch).shuffle(order)

        for shard in iter_shards(order, repo_id=repo_id, local_dir=shard_dir,
                                 keep_files=keep_files):
            dataset = TensorDataset(shard.states, shard.policy, shard.value.unsqueeze(-1), shard.masks)
            dataloader = DataLoader(
                dataset,
                batch_size=batch_size,
                shuffle=True,
                num_workers=num_workers,
                pin_memory=torch.cuda.is_available(),
                drop_last=True,
            )
            print(f"[epoch {epoch}] {shard.name}: {len(dataset)} rows, {len(dataloader)} batches")

            for batch_states, batch_policy, batch_value, batch_masks in dataloader:
                batch_states = batch_states.to(DEVICE, non_blocking=True)
                batch_policy = batch_policy.to(DEVICE, non_blocking=True)
                batch_value  = batch_value.to(DEVICE, non_blocking=True)
                batch_masks  = batch_masks.to(DEVICE, non_blocking=True)

                optimizer.zero_grad()
                policy_head, value_head = network(batch_states)
                policy_head = policy_head.masked_fill(~batch_masks, float("-inf"))

                policy_loss = policy_loss_fn(policy_head, batch_policy)
                value_loss  = value_loss_fn(value_head, batch_value)
                loss = policy_loss + value_loss
                loss.backward()

                optimizer.step()
                if step < total_steps:
                    scheduler.step()

                if step % log_every == 0:
                    elapsed = time.time() - start
                    print(f"[{step}] loss={loss.item():.4f} | policy={policy_loss.item():.4f} | value={value_loss.item():.4f} | lr: {scheduler.get_last_lr()[0]:.8f} | {elapsed:.0f}s")

                step += 1

                if duration_hour is not None and time.time() - start >= duration_hour * 3600:
                    out_of_time = True
                    break

            # Only a shard trained end to end counts as done; one cut short by
            # the session timer is left for the next session to redo.
            if not out_of_time:
                consumed.add(shard.name)

            # Validation loss stays as an informational metric.
            if has_test:
                val_policy_loss, val_value_loss = evaluate(
                    network, test_states, test_policy, test_value, test_masks,
                    policy_loss_fn, value_loss_fn,
                )
                print(f"    [eval @ {step}] val_policy={val_policy_loss:.4f} | val_value={val_value_loss:.4f} | total_val={val_policy_loss + val_value_loss:.4f}")

            # Head-to-head match every iter (per shard): promote + push to HF only
            # when the current net beats the running best. A match plays eval_games
            # full MCTS games, so this is the expensive part of the loop.
            result = match_evaluate(
                network, best_network, game=game,
                num_rollout=eval_rollout, num_games=eval_games,
                win_rate_threshold=win_rate_threshold,
            )
            print(f"    [match @ {step}] new {result.new_wins}-{result.old_wins}-{result.draws} old | win_rate {result.win_rate:.2f} | promote={result.promote}")

            promoted = result.promote
            if promoted:
                best_network = copy.deepcopy(unwrap(network))
                save_network(network, game=game, version=version, file_name=file_name,
                             parent_dir=parent_dir, path=save_path,
                             push_to_hf=push_to_hf, repo_id=push_repo_id)

            # match_evaluate leaves the net in eval mode; restore train mode.
            network.train()

            # Checkpoint on a timer, not on promotion: a session that dies
            # between promotions would otherwise lose everything since the last one.
            # The local write always happens; only a net that just won its match
            # earns the upload, so a losing net never overwrites the Hub's state.
            if out_of_time or time.time() - last_checkpoint >= checkpoint_every_min * 60:
                save_training_state(state_path, network, best_network, optimizer, scheduler,
                                    step=step, epoch=epoch, consumed=consumed, total_steps=total_steps,
                                    push_to_hf=push_to_hf and promoted, repo_id=push_repo_id,
                                    path_in_repo=state_in_repo)
                last_checkpoint = time.time()
                print(f"    [checkpoint @ {step}] epoch {epoch}, {len(consumed)}/{len(train_shards)} shards done"
                      f"{'' if promoted else ' (local only, last match not won)'}")

            if out_of_time:
                break

        # Epoch finished cleanly, so the next one starts from the full shard list.
        if not out_of_time:
            consumed = set()

    save_training_state(state_path, network, best_network, optimizer, scheduler,
                        step=step, epoch=epoch, consumed=consumed, total_steps=total_steps,
                        push_to_hf=push_to_hf and promoted, repo_id=push_repo_id,
                        path_in_repo=state_in_repo)

    print(f"done: {step} steps in {time.time() - start:.0f}s "
          f"({len(consumed)}/{len(train_shards)} shards into epoch {epoch})")

In [ ]:
train(
    duration_hour=8,   # this session only; the timer stops the run
    total_hour=48,      # planned total across sessions, shapes the LR schedule
    network=network,
    optimizer=optimizer,
    train_shards=TRAIN_SHARDS,
    test_shards=TEST_SHARDS,
    policy_loss_fn=policy_loss_fn,
    value_loss_fn=value_loss_fn,
    batch_size=512,
    keep_files=KEEP_FILES,
    train_rows=TRAIN_ROWS,
    file_name="pretrained",
    push_repo_id="williamalxndr/chess-trained-network",
    eval_games=4,
    eval_rollout=100,
    win_rate_threshold=0.55,
    resume=True,             # set False to start over from random weights
    checkpoint_every_min=20,
)